In [1]:
# 2. Importar las librerías necesarias
import pandas as pd
import numpy as np

# Configuración opcional para mostrar números flotantes con 2 decimales
pd.options.display.float_format = '{:.2f}'.format

In [2]:
# 3. Cargar los datos (rutas absolutas solicitadas) y limpiar
# Usamos las rutas exactas indicadas en los requisitos
ruta_ventas = 'workspace/sales.csv'
ruta_inventarios = 'workspace/inventories.csv'
ruta_satisfaccion = 'workspace/satisfaction.csv'

# Cargar archivos en DataFrames
df_sales = pd.read_csv(ruta_ventas)
df_inventories = pd.read_csv(ruta_inventarios)
df_satisfaction = pd.read_csv(ruta_satisfaccion)

# Limpiar los datos eliminando filas con valores nulos
df_sales = df_sales.dropna()
df_inventories = df_inventories.dropna()
df_satisfaction = df_satisfaction.dropna()

print("Datos cargados y limpiados exitosamente.")

Datos cargados y limpiados exitosamente.


In [3]:
# 4. Exploración de datos (Pandas)

# a) Ventas totales por producto y por tienda
ventas_prod_tienda = df_sales.groupby(['ID_Tienda', 'Producto'])['Cantidad_Vendida'].sum().reset_index()
ventas_tienda = df_sales.groupby('ID_Tienda')['Cantidad_Vendida'].sum().reset_index()
ventas_producto = df_sales.groupby('Producto')['Cantidad_Vendida'].sum().reset_index()
print("\n--- Ventas Totales por Tienda ---")
display(ventas_tienda)
print("\n--- Ventas Totales por Producto ---")
display(ventas_producto)
print("--- Ventas Totales por Tienda y Producto ---")
display(ventas_prod_tienda.head())

# b) Ingresos totales por tienda
# Creamos la columna 'Total_Ingresos' (Ingresos) multiplicando Cantidad por Precio Unitario
df_sales['Total_Ingresos'] = df_sales['Cantidad_Vendida'] * df_sales['Precio_Unitario']
ingresos_tienda = df_sales.groupby('ID_Tienda')['Total_Ingresos'].sum().reset_index()
print("\n--- Ingresos Totales por Tienda ---")
display(ingresos_tienda)

# c) Resumen estadístico
print("\n--- Resumen Estadístico de Ventas ---")
display(df_sales.describe())

# d) Promedio de ventas por tienda y categoría de productos
promedio_ventas_tienda = df_sales.groupby('ID_Tienda')['Cantidad_Vendida'].mean().reset_index()
print("\n--- Promedio de Ventas por Tienda ---")
display(promedio_ventas_tienda)
promedio_ventas_producto = df_sales.groupby('Producto')['Cantidad_Vendida'].mean().reset_index()
print("\n--- Promedio de Ventas por Producto ---")
display(promedio_ventas_producto)


--- Ventas Totales por Tienda ---


,ID_Tienda,Cantidad_Vendida
0,1,35
1,2,55
2,3,50
3,4,60
4,5,50



--- Ventas Totales por Producto ---


,Producto,Cantidad_Vendida
0,Producto A,85
1,Producto B,75
2,Producto C,90


--- Ventas Totales por Tienda y Producto ---


...

,ID_Tienda,Cantidad_Vendida,Precio_Unitario,Total_Ingresos
count,10.00,10.00,10.00,10.00
mean,3.00,25.00,190.00,5050.00
std,1.49,9.13,87.56,3361.96
min,1.00,10.00,100.00,1000.00
25%,2.00,20.00,100.00,2625.00
50%,3.00,25.00,200.00,3500.00
75%,4.00,30.00,275.00,7875.00
max,5.00,40.00,300.00,10500.00



--- Promedio de Ventas por Tienda ---


,ID_Tienda,Cantidad_Vendida
0,1,17.50
1,2,27.50
2,3,25.00
3,4,30.00
4,5,25.00



--- Promedio de Ventas por Producto ---


,Producto,Cantidad_Vendida
0,Producto A,21.25
1,Producto B,25.00
2,Producto C,30.00


In [4]:
# 5. Análisis de inventarios (Pandas)

# Agrupamos las ventas totales por tienda y producto usando los nombres correctos
ventas_agrupadas = df_sales.groupby(['ID_Tienda', 'Producto'])['Cantidad_Vendida'].sum().reset_index()

# Unimos los dataframes de inventarios y ventas agrupadas
df_inv_analisis = pd.merge(df_inventories, ventas_agrupadas, on=['ID_Tienda', 'Producto'], how='left')

# Si un producto en inventario no tuvo ventas, rellenamos con 0 para poder hacer el cálculo matemático
df_inv_analisis['Cantidad_Vendida'] = df_inv_analisis['Cantidad_Vendida'].fillna(0)

# a) Calcular la rotación de inventarios (Ventas Totales / Stock)
# Utilizamos la columna exacta del archivo: 'Stock_Disponible'
df_inv_analisis['Rotacion_Inventario'] = df_inv_analisis['Cantidad_Vendida'] / df_inv_analisis['Stock_Disponible']

# b) Filtrar tiendas con niveles críticos (< 10% es decir, < 0.10)
inventario_critico = df_inv_analisis[df_inv_analisis['Rotacion_Inventario'] < 0.10]
print("análisis de inventario")
display(df_inv_analisis)
print("--- Tiendas y Productos con Niveles Críticos de Inventario (< 10% rotación) ---")
display(inventario_critico)

análisis de inventario


,ID_Tienda,Producto,Stock_Disponible,Fecha_Actualización,Cantidad_Vendida,Rotacion_Inventario
0,1,Producto A,50,2023-01-05,20,0.40
1,1,Producto B,40,2023-01-06,15,0.38
2,2,Producto A,60,2023-01-07,30,0.50
3,2,Producto C,45,2023-01-08,25,0.56
4,3,Producto A,30,2023-01-09,10,0.33
5,3,Producto B,80,2023-01-10,40,0.50
6,4,Producto C,70,2023-01-11,35,0.50
7,4,Producto A,50,2023-01-12,25,0.50
8,5,Producto B,40,2023-01-13,20,0.50
9,5,Producto C,60,2023-01-14,30,0.50


--- Tiendas y Productos con Niveles Críticos de Inventario (< 10% rotación) ---


,ID_Tienda,Producto,Stock_Disponible,Fecha_Actualización,Cantidad_Vendida,Rotacion_Inventario


In [5]:
# 6. Satisfacción del cliente (Pandas)

# Unimos la satisfacción con los ingresos totales calculados en el paso 4 usando 'ID_Tienda'
# La columna exacta del archivo se llama 'Satisfacción_Promedio'
df_rendimiento = pd.merge(ingresos_tienda, df_satisfaction, on='ID_Tienda', how='inner')

# Filtrar tiendas con baja satisfacción (< 60%)
tiendas_baja_sat = df_rendimiento[df_rendimiento['Satisfacción_Promedio'] < 60]

print("--- Tiendas con Baja Satisfacción (< 60%) ---")
display(tiendas_baja_sat)

--- Tiendas con Baja Satisfacción (< 60%) ---


,ID_Tienda,Total_Ingresos,Satisfacción_Promedio,Fecha_Evaluación
4,5,13000,55,2023-01-15



Recomendaciones:
* Mejorar la atención al cliente
* Hacer procesos de seguimiento periódicos a la atención

In [6]:
# 7. Operaciones con Numpy

# a) Convertir la columna Total_Ventas (creada en el paso 4) a un array de Numpy
Total_Ventas = df_sales['Cantidad_Vendida'].to_numpy()
Total_Ingresos = df_sales['Total_Ingresos'].to_numpy()

# b) Calcular Mediana y Desviación Estándar de las ventas totales
mediana_ventas = np.median(Total_Ventas)
desviacion_ventas = np.std(Total_Ventas)

mediana_ingresos = np.median(Total_Ingresos)
desviacion_ingresos = np.std(Total_Ingresos)

print(f"Mediana de ventas totales (por transacción): ${mediana_ventas:,.2f}")
print(f"Desviación estándar de ventas (por transacción): ${desviacion_ventas:,.2f}")
print(f"Mediana de ingresos totales (por transacción): ${mediana_ingresos:,.2f}")
print(f"Desviación estándar de ingresos (por transacción): ${desviacion_ingresos:,.2f}")

# c) Simular proyecciones de ventas futuras
# Establecemos una semilla para resultados reproducibles
np.random.seed(42)

# Simulamos un escenario donde las ventas futuras fluctúan aleatoriamente 
# entre una caída del 5% y un aumento del 15% respecto a las ventas actuales
factor_crecimiento = np.random.uniform(low=-0.05, high=0.15, size=len(Total_Ventas))
ventas_proyectadas = Total_Ventas * (1 + factor_crecimiento)
factor_crecimiento = np.random.uniform(low=-0.05, high=0.15, size=len(Total_Ventas))
ingresos_proyectados = Total_Ingresos * (1 + factor_crecimiento)
# Mostrar un resumen de la proyección
print("\n--- Simulación de Proyecciones Futuras de ventas ---")
print(f"Ventas totales actuales: ${np.sum(Total_Ventas):,.2f}")
print(f"Ventas totales proyectadas: ${np.sum(ventas_proyectadas):,.2f}")
print(f"Variación porcentual global: {((np.sum(ventas_proyectadas) / np.sum(Total_Ventas)) - 1) * 100:.2f}%")

print("\n--- Simulación de Proyecciones Futuras de ingresos ---")
print(f"Ingresos totales actuales: ${np.sum(Total_Ingresos):,.2f}")
print(f"Ingresos totales proyectados: ${np.sum(ingresos_proyectados):,.2f}")
print(f"Variación porcentual global: {((np.sum(ingresos_proyectados) / np.sum(Total_Ingresos)) - 1) * 100:.2f}%")

Mediana de ventas totales (por transacción): $25.00
Desviación estándar de ventas (por transacción): $8.66
Mediana de ingresos totales (por transacción): $3,500.00
Desviación estándar de ingresos (por transacción): $3,189.44

--- Simulación de Proyecciones Futuras de ventas ---
Ventas totales actuales: $250.00
Ventas totales proyectadas: $262.19
Variación porcentual global: 4.87%

--- Simulación de Proyecciones Futuras de ingresos ---
Ingresos totales actuales: $50,500.00
Ingresos totales proyectados: $51,484.02
Variación porcentual global: 1.95%
